# Expanded Dataset Baseline Models

## Objective

This notebook evaluates baseline machine learning models using the expanded human-annotated Frankenstein dataset.

The goal is to compare model performance after increasing annotation coverage from the pilot dataset to an expanded supervised dataset.

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report
)

print("Libraries loaded successfully")

Libraries loaded successfully


## Loading Expanded Annotated Dataset

The dataset contains human-annotated Frankenstein passages with therapeutic labels, mechanisms, intensity scores, and annotation notes.

In [2]:
from google.colab import files

uploaded = files.upload()

Saving frankenstein_training_dataset.csv to frankenstein_training_dataset.csv


In [3]:
df = pd.read_csv("frankenstein_training_dataset.csv")

print("Dataset loaded successfully")
print("Dataset shape:", df.shape)

df.head()

Dataset loaded successfully
Dataset shape: (29, 7)


,passage_id,text,therapeutic_label,mechanism,intensity,annotator_notes,keyword_score
0,10,This appearance excited our unqualified wonder...,0.0,NaN,0.0,The passage describes curiosity and wonder dur...,NaN
1,16,Will you smile at the enthusiasm I express con...,1.0,Narrative Reconstruction,3.0,The passage introduces Victor's suffering and ...,NaN
2,26,I feel exquisite pleasure in dwelling on the r...,1.0,Narrative Reconstruction,4.0,The passage represents autobiographical reflec...,NaN
3,67,"“She most of all,” said Ernest, “requires cons...",0.0,NaN,0.0,"The passage describes grief, guilt, and emotio...",NaN
4,127,“I knocked. ‘Who is there?’ said the old man. ...,0.0,NaN,0.0,The passage represents a compassionate social ...,NaN


In [4]:
df["therapeutic_label"].value_counts()

,count
therapeutic_label,
0.0,18
1.0,11


## Preparing Text Features and Labels

The literary passages are converted into numerical TF-IDF features for machine learning classification.

The therapeutic_label column is used as the prediction target.

In [5]:
X = df["text"]

y = df["therapeutic_label"]

print("Input samples:", len(X))
print("Labels:", len(y))

y.value_counts()

Input samples: 29
Labels: 29


,count
therapeutic_label,
0.0,18
1.0,11


In [6]:
tfidf = TfidfVectorizer(
    max_features=1000,
    stop_words="english"
)

X_features = tfidf.fit_transform(X)

print("Feature shape:", X_features.shape)

Feature shape: (29, 1000)


In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X_features,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])

Training samples: 23
Testing samples: 6


## Logistic Regression Baseline Model

Logistic Regression is used as the first baseline classifier to predict whether a passage represents a therapeutic process.

In [8]:
log_model = LogisticRegression(
    random_state=42
)

log_model.fit(
    X_train,
    y_train
)

print("Logistic Regression model trained successfully")

Logistic Regression model trained successfully


In [9]:
y_pred = log_model.predict(X_test)

print("Predicted:", y_pred)
print("Actual:", y_test.values)

Predicted: [0. 0. 0. 0. 0. 0.]
Actual: [1. 0. 0. 1. 0. 0.]


In [10]:
accuracy = accuracy_score(
    y_test,
    y_pred
)

print("Logistic Regression Accuracy:", accuracy)

Logistic Regression Accuracy: 0.6666666666666666


In [11]:
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_pred,
        zero_division=0
    )
)

Confusion Matrix:
[[4 0]
 [2 0]]

Classification Report:
              precision    recall  f1-score   support

         0.0       0.67      1.00      0.80         4
         1.0       0.00      0.00      0.00         2

    accuracy                           0.67         6
   macro avg       0.33      0.50      0.40         6
weighted avg       0.44      0.67      0.53         6



## Random Forest Baseline Model

Random Forest is evaluated as a second baseline classifier on the expanded annotated dataset.

In [12]:
rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

rf_model.fit(
    X_train,
    y_train
)

print("Random Forest model trained successfully")

Random Forest model trained successfully


In [13]:
rf_pred = rf_model.predict(X_test)

print("Predicted:", rf_pred)
print("Actual:", y_test.values)

Predicted: [0. 0. 0. 0. 0. 0.]
Actual: [1. 0. 0. 1. 0. 0.]


In [14]:
print("Random Forest Accuracy:",
      accuracy_score(y_test, rf_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, rf_pred))

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        rf_pred,
        zero_division=0
    )
)

Random Forest Accuracy: 0.6666666666666666

Confusion Matrix:
[[4 0]
 [2 0]]

Classification Report:
              precision    recall  f1-score   support

         0.0       0.67      1.00      0.80         4
         1.0       0.00      0.00      0.00         2

    accuracy                           0.67         6
   macro avg       0.33      0.50      0.40         6
weighted avg       0.44      0.67      0.53         6



## Expanded Baseline Model Results

The expanded dataset improved accuracy compared with the pilot experiment. However, both Logistic Regression and Random Forest showed difficulty identifying therapeutic passages due to the limited number of positive examples.

Further annotation expansion is required to improve minority-class recognition.

## Error Analysis

This section examines incorrectly classified passages to understand model limitations and identify patterns requiring further annotation or feature improvement.

In [15]:
error_analysis = pd.DataFrame({
    "text": df.loc[y_test.index, "text"],
    "actual": y_test,
    "predicted": y_pred
})

errors = error_analysis[
    error_analysis["actual"] != error_analysis["predicted"]
]

errors

,text,actual,predicted
2,I feel exquisite pleasure in dwelling on the r...,1.0,0.0
9,"My beloved Sister, September 2d. I write to yo...",1.0,0.0


In [16]:
for i, row in errors.iterrows():
    print("\nPASSAGE ID:", df.loc[i, "passage_id"])
    print("Actual:", row["actual"])
    print("Predicted:", row["predicted"])
    print(row["text"][:500])
    print("-"*80)


PASSAGE ID: 26
Actual: 1.0
Predicted: 0.0
I feel exquisite pleasure in dwelling on the recollections of childhood,
before misfortune had tainted my mind and changed its bright visions of
extensive usefulness into gloomy and narrow reflections upon self. Besides,
in drawing the picture of my early days, I also record those events which
led, by insensible steps, to my after tale of misery, for when I would
account to myself for the birth of that passion which afterwards ruled my
destiny I find it arise, like a mountain river, from ignoble
--------------------------------------------------------------------------------

PASSAGE ID: 217
Actual: 1.0
Predicted: 0.0
My beloved Sister, September 2d. I write to you, encompassed by peril and ignorant whether I am ever
doomed to see again dear England and the dearer friends that inhabit
it. I am surrounded by mountains of ice which admit of no escape and
threaten every moment to crush my vessel. The brave fellows whom I
have persuaded to be my c

## Error Analysis Findings

The baseline models incorrectly classified therapeutic passages containing strong negative emotional vocabulary.

Passage 26 was misclassified because the model focused on words associated with suffering, while missing the narrative reconstruction process through autobiographical reflection.

Passage 217 was misclassified because the model identified fear and danger-related terms but failed to recognize emotional disclosure, hope, and interpersonal communication.

These errors demonstrate that TF-IDF based models capture lexical patterns but struggle with deeper narrative and therapeutic functions.

In [17]:
feature_names = tfidf.get_feature_names_out()

coefficients = log_model.coef_[0]

feature_importance = pd.DataFrame({
    "word": feature_names,
    "importance": coefficients
})

feature_importance.sort_values(
    by="importance",
    ascending=False
).head(15)

,word,importance
160,dear,0.270695
138,countenance,0.205539
666,lady,0.199818
179,despair,0.187525
187,die,0.179952
59,beauty,0.174672
954,voice,0.168736
448,friend,0.164937
382,feel,0.161689
932,true,0.155297


In [18]:
feature_importance.sort_values(
    by="importance",
    ascending=True
).head(15)

,word,importance
743,night,-0.215537
718,miserable,-0.202493
583,ice,-0.171697
348,eyes,-0.170918
669,language,-0.129378
961,wedding,-0.127841
446,french,-0.119455
728,morning,-0.116056
724,monster,-0.114946
939,union,-0.113386


## Feature Importance Analysis

Logistic Regression feature analysis shows that the model relies heavily on lexical associations.

Words related to interpersonal connection, emotional expression, and reflection such as "dear", "friend", "affection", and "sympathy" contributed positively toward therapeutic classification.

However, words associated with suffering, danger, and isolation such as "night", "monster", "ice", and "miserable" were associated with non-therapeutic classification.

The results demonstrate that TF-IDF based models identify surface-level linguistic patterns but have difficulty distinguishing between emotional suffering and therapeutic transformation. This limitation highlights the need for larger annotated datasets and more context-aware NLP approaches.